In [42]:
import pandas as pd
import torch
from torch.utils.data import Dataset,DataLoader

In [16]:
df = pd.read_csv("/content/100_Unique_QA_Dataset.csv")
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [25]:
# TOKENIZE
def tokenize(text):
  text = text.lower()
  text = text.replace("?",'')
  text = text.replace("!",'')
  text = text.replace(".","")
  text = text.replace(",","")
  text = text.replace("-",'')

  return text.split()

In [26]:
vocab = {'<UNK>':0}


In [35]:
def build_vocab(row):
  print(row['question'],row['answer'])
  tokenized_question = tokenize(row['question'])
  tokenized_answer = tokenize(row['answer'])
  merged_token = tokenized_question + tokenized_answer
  print(merged_token)

  for token in merged_token:
    if token not in vocab:
      vocab[token] = len(vocab)

In [36]:
df.apply(build_vocab,axis=1)

What is the capital of France? Paris
['what', 'is', 'the', 'capital', 'of', 'france', 'paris']
What is the capital of Germany? Berlin
['what', 'is', 'the', 'capital', 'of', 'germany', 'berlin']
Who wrote 'To Kill a Mockingbird'? Harper-Lee
['who', 'wrote', "'to", 'kill', 'a', "mockingbird'", 'harperlee']
What is the largest planet in our solar system? Jupiter
['what', 'is', 'the', 'largest', 'planet', 'in', 'our', 'solar', 'system', 'jupiter']
What is the boiling point of water in Celsius? 100
['what', 'is', 'the', 'boiling', 'point', 'of', 'water', 'in', 'celsius', '100']
Who painted the Mona Lisa? Leonardo-da-Vinci
['who', 'painted', 'the', 'mona', 'lisa', 'leonardodavinci']
What is the square root of 64? 8
['what', 'is', 'the', 'square', 'root', 'of', '64', '8']
What is the chemical symbol for gold? Au
['what', 'is', 'the', 'chemical', 'symbol', 'for', 'gold', 'au']
Which year did World War II end? 1945
['which', 'year', 'did', 'world', 'war', 'ii', 'end', '1945']
What is the longes

,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [37]:
vocab

{'<UNK>': 0,
 'token': 2,
 'what': 2,
 'is': 3,
 'the': 4,
 'capital': 5,
 'of': 6,
 'france': 7,
 'paris': 8,
 'germany': 9,
 'berlin': 10,
 'who': 11,
 'wrote': 12,
 "'to": 13,
 'kill': 14,
 'a': 15,
 "mockingbird'": 16,
 'harperlee': 17,
 'largest': 18,
 'planet': 19,
 'in': 20,
 'our': 21,
 'solar': 22,
 'system': 23,
 'jupiter': 24,
 'boiling': 25,
 'point': 26,
 'water': 27,
 'celsius': 28,
 '100': 29,
 'painted': 30,
 'mona': 31,
 'lisa': 32,
 'leonardodavinci': 33,
 'square': 34,
 'root': 35,
 '64': 36,
 '8': 37,
 'chemical': 38,
 'symbol': 39,
 'for': 40,
 'gold': 41,
 'au': 42,
 'which': 43,
 'year': 44,
 'did': 45,
 'world': 46,
 'war': 47,
 'ii': 48,
 'end': 49,
 '1945': 50,
 'longest': 51,
 'river': 52,
 'nile': 53,
 'japan': 54,
 'tokyo': 55,
 'developed': 56,
 'theory': 57,
 'relativity': 58,
 'alberteinstein': 59,
 'freezing': 60,
 'fahrenheit': 61,
 '32': 62,
 'known': 63,
 'as': 64,
 'red': 65,
 'mars': 66,
 'author': 67,
 "'1984'": 68,
 'georgeorwell': 69,
 'currency

In [40]:
# CONVERT TEXT TO INDICIES
def text_to_indices(text,vocab):
  index_text = []
  for token in tokenize(text):
     if token in vocab:
       index_text.append(vocab[token])
     else:
        index_text.append(vocab['<UNK>'])
  return index_text

In [41]:
text_to_indices("what is are",vocab)

[2, 3, 82]

In [43]:
class QAdataset(Dataset):
  def __init__(self,df,vocab):
    self.df = df
    self.vocab = vocab

  def __len__(self):
    return self.df.shape[0]

  def __getitem__(self,index):
    numerical_question = text_to_indices(self.df.iloc[index]['question'],self.vocab)
    numerical_answer = text_to_indices(self.df.iloc[index]['answer'],self.vocab)

    return torch.tensor(numerical_question),torch.tensor(numerical_answer)

In [45]:
dataset = QAdataset(df,vocab)
dataset[7]

(tensor([ 2,  3,  4, 38, 39, 40, 41]), tensor([42]))

In [47]:
dataloader = DataLoader(dataset,batch_size=1,shuffle=True)

In [48]:
import torch.nn as nn

In [54]:
class SimpleRNN(nn.Module):
    def __init__(self,vocab_size):
       super().__init__()
       self.embedding = nn.Embedding(vocab_size,embedding_dim=50)
       self.rnn = nn.RNN(50,64,batch_first=True)
       self.fc = nn.Linear(64,vocab_size)

    def forward(self,question):
      embedded_question = self.embedding(question)
      hidden,final = self.rnn(embedded_question)
      output = self.fc(final.squeeze(0))
      return output

In [55]:
learning_rate = 0.01
epochs= 20

In [56]:
model = SimpleRNN(len(vocab))


In [57]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=learning_rate)

In [66]:

for epoch in range(epochs):
    total_loss = 0
    for question , answer in dataloader:
      optimizer.zero_grad()

      output = model(question)

      loss= criterion(output,answer[0])
      loss.backward()

      optimizer.step()

      total_loss = total_loss + loss.item()

    print(f"{epoch+1},Loss:{total_loss:4f}")

1,Loss:0.380725
2,Loss:0.338316
3,Loss:0.305191
4,Loss:0.278125
5,Loss:0.255036
6,Loss:0.235250
7,Loss:0.217942
8,Loss:0.201413
9,Loss:0.187947
10,Loss:0.175160
11,Loss:0.163677
12,Loss:0.153002
13,Loss:0.143497
14,Loss:0.134968
15,Loss:0.126489
16,Loss:0.119312
17,Loss:0.112378
18,Loss:0.105838
19,Loss:0.100080
20,Loss:0.094769


In [72]:
def predict(model,question,threshold=0.5):
  numercial_question = text_to_indices(question,vocab)
  question_tensor = torch.tensor(numercial_question).unsqueeze(0)
  output = model(question_tensor)
  probs = torch.nn.functional.softmax(output,dim = 1)
  value , index =  torch.max(probs,dim =1)
  if value < threshold:
    print("I DONOT KNOW")
  else:
    print(list(vocab.keys())[index])

In [74]:
predict(model,"what is capital of france")

paris


In [68]:
list(vocab.keys())[7]

'france'